In [32]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import librosa
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

In [33]:
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

PyTorch: 2.5.1+cu121
CUDA available: True
Using device: cuda


In [34]:
df = pd.read_csv(r"C:\Users\Arup Sarkar\Documents\EmoWave\data\dataemotion_data.csv")

N_MELS = 128
SR = 22050
DURATION = 3
N_FFT = 2048
HOP_LENGTH = 512
NUM_CLASSES = 8

print(f'Total clips: {len(df)}')
print(f'Classes: {df["emotion"].nunique()}')

Total clips: 1440
Classes: 8


In [35]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['emotion'])

class EmoWaveDataset(Dataset):
    def __init__(self, df, augment_data=False):
        self.df = df.reset_index(drop=True)
        self.augment_data = augment_data

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = row['path']

        y, sr = librosa.load(file_path, sr=SR, duration=DURATION)

        if len(y) < SR * DURATION:
            y = np.pad(y, (0, SR * DURATION - len(y)))

        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS,
                                           n_fft=N_FFT, hop_length=HOP_LENGTH)
        S_db = librosa.power_to_db(S, ref=np.max)

        S_db = (S_db - S_db.mean()) / (S_db.std() + 1e-8)

        tensor = torch.tensor(S_db, dtype=torch.float32).unsqueeze(0)

        label = torch.tensor(row['label'], dtype=torch.long)

        if self.augment_data:
            y = self.augment(y, sr)

        return tensor, label

    def augment(self, y, sr):
    # Add random noise
        noise = np.random.randn(len(y)) * 0.005
        y = y + noise

    # Time shift
        shift = np.random.randint(sr // 4)
        y = np.roll(y, shift)

    # Pitch shift
        steps = np.random.uniform(-2, 2)
        y = librosa.effects.pitch_shift(y, sr=sr, n_steps=steps)

        return y

print('Dataset class defined')

Dataset class defined


In [36]:
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)

train_dataset = EmoWaveDataset(train_df, augment_data=True)
test_dataset = EmoWaveDataset(test_df, augment_data=False)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f'Train samples: {len(train_dataset)}')
print(f'Test samples:  {len(test_dataset)}')
print(f'Train batches: {len(train_loader)}')
print(f'Test batches:  {len(test_loader)}')

# verify one batch
batch_x, batch_y = next(iter(train_loader))
print(f'Batch X shape: {batch_x.shape}')
print(f'Batch y shape: {batch_y.shape}')

Train samples: 1152
Test samples:  288
Train batches: 36
Test batches:  9
Batch X shape: torch.Size([32, 1, 128, 130])
Batch y shape: torch.Size([32])


In [37]:
dummy = torch.zeros(1, 1, 128, 130).to(device)

In [38]:
class EmoWaveNet(nn.Module):
    def __init__(self, num_classes=8):
        super(EmoWaveNet, self).__init__()

        self.conv_block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.conv_block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.classifier(x)
        return x

model = EmoWaveNet(num_classes=NUM_CLASSES).to(device)

In [39]:
with torch.no_grad():
    dummy = torch.zeros(1, 1, 128, 130).to(device)
    out = model.conv_block1(dummy)
    out = model.conv_block2(out)
    out = model.conv_block3(out)
    print(out.shape)
    print(out.view(1, -1).shape[1])

torch.Size([1, 128, 16, 16])
32768


In [40]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

EPOCHS = 30
train_losses = []
test_accuracies = []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()

    acc = correct / total
    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)
    test_accuracies.append(acc)

    print(f'Epoch [{epoch+1}/{EPOCHS}] Loss: {avg_loss:.4f} | Test Acc: {acc:.4f}')

Epoch [1/30] Loss: 7.2466 | Test Acc: 0.2465
Epoch [2/30] Loss: 1.8956 | Test Acc: 0.2917
Epoch [3/30] Loss: 1.8168 | Test Acc: 0.3090
Epoch [4/30] Loss: 1.8246 | Test Acc: 0.3056
Epoch [5/30] Loss: 1.7717 | Test Acc: 0.3368
Epoch [6/30] Loss: 1.7617 | Test Acc: 0.3368
Epoch [7/30] Loss: 1.7137 | Test Acc: 0.3472
Epoch [8/30] Loss: 1.6731 | Test Acc: 0.3924
Epoch [9/30] Loss: 1.6636 | Test Acc: 0.3750
Epoch [10/30] Loss: 1.6395 | Test Acc: 0.3646
Epoch [11/30] Loss: 1.5726 | Test Acc: 0.4097
Epoch [12/30] Loss: 1.5223 | Test Acc: 0.4201
Epoch [13/30] Loss: 1.4490 | Test Acc: 0.4236
Epoch [14/30] Loss: 1.4151 | Test Acc: 0.4444
Epoch [15/30] Loss: 1.3953 | Test Acc: 0.4340
Epoch [16/30] Loss: 1.3277 | Test Acc: 0.4479
Epoch [17/30] Loss: 1.3145 | Test Acc: 0.4861
Epoch [18/30] Loss: 1.3412 | Test Acc: 0.4306
Epoch [19/30] Loss: 1.2683 | Test Acc: 0.4479
Epoch [20/30] Loss: 1.2314 | Test Acc: 0.4861
Epoch [21/30] Loss: 1.1784 | Test Acc: 0.5069
Epoch [22/30] Loss: 1.1575 | Test Acc: 0.52

In [41]:
EPOCHS2 = 30

for epoch in range(EPOCHS2):
    model.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()

    acc = correct / total
    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)
    test_accuracies.append(acc)

    print(f'Epoch [{epoch+31}/{60}] Loss: {avg_loss:.4f} | Test Acc: {acc:.4f}')

Epoch [31/60] Loss: 0.9943 | Test Acc: 0.5521
Epoch [32/60] Loss: 0.9910 | Test Acc: 0.5382
Epoch [33/60] Loss: 0.9629 | Test Acc: 0.5486
Epoch [34/60] Loss: 0.9636 | Test Acc: 0.5590
Epoch [35/60] Loss: 0.9248 | Test Acc: 0.5278
Epoch [36/60] Loss: 0.9403 | Test Acc: 0.5417
Epoch [37/60] Loss: 0.9067 | Test Acc: 0.5451
Epoch [38/60] Loss: 0.9056 | Test Acc: 0.5486
Epoch [39/60] Loss: 0.9131 | Test Acc: 0.5694
Epoch [40/60] Loss: 0.8614 | Test Acc: 0.5764
Epoch [41/60] Loss: 0.8576 | Test Acc: 0.5660
Epoch [42/60] Loss: 0.8496 | Test Acc: 0.5694
Epoch [43/60] Loss: 0.8265 | Test Acc: 0.5417
Epoch [44/60] Loss: 0.8535 | Test Acc: 0.5556
Epoch [45/60] Loss: 0.8225 | Test Acc: 0.5660
Epoch [46/60] Loss: 0.8396 | Test Acc: 0.5694
Epoch [47/60] Loss: 0.7953 | Test Acc: 0.5799
Epoch [48/60] Loss: 0.7966 | Test Acc: 0.5694
Epoch [49/60] Loss: 0.8221 | Test Acc: 0.5486
Epoch [50/60] Loss: 0.8042 | Test Acc: 0.5799
Epoch [51/60] Loss: 0.8006 | Test Acc: 0.5764
Epoch [52/60] Loss: 0.7785 | Test 

In [42]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import torchvision.models as models
efficientnet = models.efficientnet_b0(weights='IMAGENET1K_V1')
print(efficientnet.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)


In [43]:
# Replace final layer for 8 classes
efficientnet.classifier = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(1280, NUM_CLASSES)
)

# Convert grayscale (1 channel) to 3 channels EfficientNet expects
efficientnet.features[0][0] = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)

# Move to GPU
efficientnet = efficientnet.to(device)

# Freeze all layers except classifier
for param in efficientnet.parameters():
    param.requires_grad = False

# Unfreeze classifier only
for param in efficientnet.classifier.parameters():
    param.requires_grad = True

# Unfreeze first conv too since we changed it
for param in efficientnet.features[0].parameters():
    param.requires_grad = True

print('EfficientNet ready')
print(f'Classifier: {efficientnet.classifier}')

EfficientNet ready
Classifier: Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=8, bias=True)
)


In [44]:
optimizer_eff = optim.Adam(
    filter(lambda p: p.requires_grad, efficientnet.parameters()),
    lr=0.001
)
scheduler_eff = optim.lr_scheduler.StepLR(optimizer_eff, step_size=10, gamma=0.5)

EPOCHS_EFF = 30
eff_losses = []
eff_accuracies = []

for epoch in range(EPOCHS_EFF):
    efficientnet.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer_eff.zero_grad()
        outputs = efficientnet(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer_eff.step()
        running_loss += loss.item()

    scheduler_eff.step()

    efficientnet.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = efficientnet(X_batch)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()

    acc = correct / total
    avg_loss = running_loss / len(train_loader)
    eff_losses.append(avg_loss)
    eff_accuracies.append(acc)

    print(f'Epoch [{epoch+1}/{EPOCHS_EFF}] Loss: {avg_loss:.4f} | Test Acc: {acc:.4f}')

Epoch [1/30] Loss: 1.9707 | Test Acc: 0.1597
Epoch [2/30] Loss: 1.7310 | Test Acc: 0.3333
Epoch [3/30] Loss: 1.6001 | Test Acc: 0.3542
Epoch [4/30] Loss: 1.5019 | Test Acc: 0.3646
Epoch [5/30] Loss: 1.4345 | Test Acc: 0.3785
Epoch [6/30] Loss: 1.4024 | Test Acc: 0.3750
Epoch [7/30] Loss: 1.3158 | Test Acc: 0.3681
Epoch [8/30] Loss: 1.2751 | Test Acc: 0.3924
Epoch [9/30] Loss: 1.2517 | Test Acc: 0.3576
Epoch [10/30] Loss: 1.2468 | Test Acc: 0.3472
Epoch [11/30] Loss: 1.1700 | Test Acc: 0.4028
Epoch [12/30] Loss: 1.1528 | Test Acc: 0.3924
Epoch [13/30] Loss: 1.1266 | Test Acc: 0.3785
Epoch [14/30] Loss: 1.1339 | Test Acc: 0.3993
Epoch [15/30] Loss: 1.1074 | Test Acc: 0.3924
Epoch [16/30] Loss: 1.1000 | Test Acc: 0.3889
Epoch [17/30] Loss: 1.1086 | Test Acc: 0.3993
Epoch [18/30] Loss: 1.1022 | Test Acc: 0.3785
Epoch [19/30] Loss: 1.0986 | Test Acc: 0.4097
Epoch [20/30] Loss: 1.0509 | Test Acc: 0.3785
Epoch [21/30] Loss: 1.0537 | Test Acc: 0.4167
Epoch [22/30] Loss: 1.0239 | Test Acc: 0.41

In [45]:
# Unfreeze everything
for param in efficientnet.parameters():
    param.requires_grad = True

# Lower learning rate for fine-tuning
optimizer_eff2 = optim.Adam(efficientnet.parameters(), lr=0.0001)
scheduler_eff2 = optim.lr_scheduler.StepLR(optimizer_eff2, step_size=10, gamma=0.5)

EPOCHS_EFF2 = 70
for epoch in range(EPOCHS_EFF2):
    efficientnet.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer_eff2.zero_grad()
        outputs = efficientnet(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer_eff2.step()
        running_loss += loss.item()

    scheduler_eff2.step()

    efficientnet.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = efficientnet(X_batch)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()

    acc = correct / total
    avg_loss = running_loss / len(train_loader)
    print(f'Epoch [{epoch+31}/{100}] Loss: {avg_loss:.4f} | Test Acc: {acc:.4f}')


Epoch [31/100] Loss: 1.0144 | Test Acc: 0.4306
Epoch [32/100] Loss: 0.6695 | Test Acc: 0.4514
Epoch [33/100] Loss: 0.4995 | Test Acc: 0.4861
Epoch [34/100] Loss: 0.3920 | Test Acc: 0.5035
Epoch [35/100] Loss: 0.2789 | Test Acc: 0.5278
Epoch [36/100] Loss: 0.2245 | Test Acc: 0.5208
Epoch [37/100] Loss: 0.1888 | Test Acc: 0.4688
Epoch [38/100] Loss: 0.1582 | Test Acc: 0.5139
Epoch [39/100] Loss: 0.1481 | Test Acc: 0.4965
Epoch [40/100] Loss: 0.1227 | Test Acc: 0.5243
Epoch [41/100] Loss: 0.1013 | Test Acc: 0.5174
Epoch [42/100] Loss: 0.0939 | Test Acc: 0.5243
Epoch [43/100] Loss: 0.0940 | Test Acc: 0.5312
Epoch [44/100] Loss: 0.0791 | Test Acc: 0.5139
Epoch [45/100] Loss: 0.0833 | Test Acc: 0.5174
Epoch [46/100] Loss: 0.0683 | Test Acc: 0.5104
Epoch [47/100] Loss: 0.0744 | Test Acc: 0.5278
Epoch [48/100] Loss: 0.0630 | Test Acc: 0.5556
Epoch [49/100] Loss: 0.0573 | Test Acc: 0.5590
Epoch [50/100] Loss: 0.0625 | Test Acc: 0.5417
Epoch [51/100] Loss: 0.0560 | Test Acc: 0.5486
Epoch [52/100

## RE EVALUATING MODEL ON COMBINED DATASET

In [47]:
combined_df = pd.read_csv(r"C:\Users\Arup Sarkar\Documents\EmoWave\data\combine_emotion_data.csv")

In [48]:
N_MELS = 128
SR = 22050
DURATION = 3
N_FFT = 2048
HOP_LENGTH = 512
NUM_CLASSES = 7

print(f'Total clips: {len(combined_df)}')
print(f'Classes: {combined_df["emotion"].nunique()}')

Total clips: 4048
Classes: 7


In [50]:
le = LabelEncoder()
combined_df['label'] = le.fit_transform(combined_df['emotion'])

class EmoWaveDataset(Dataset):
    def __init__(self, df, augment_data=False):
        self.df = df.reset_index(drop=True)
        self.augment_data = augment_data

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = row['path']

        y, sr = librosa.load(file_path, sr=SR, duration=DURATION)

        if len(y) < SR * DURATION:
            y = np.pad(y, (0, SR * DURATION - len(y)))

        if self.augment_data:        # ← augment BEFORE mel spectrogram
            y = self.augment(y, sr)

        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS,
                                       n_fft=N_FFT, hop_length=HOP_LENGTH)
        S_db = librosa.power_to_db(S, ref=np.max)
        S_db = (S_db - S_db.mean()) / (S_db.std() + 1e-8)

        tensor = torch.tensor(S_db, dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(row['label'], dtype=torch.long)

        return tensor, label

print('Dataset class defined')

Dataset class defined
